In [ ]:
!pip uninstall -y torch
!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!uv pip install insightface onnxruntime numpy matplotlib

Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Using Python 3.12.12 environment at: /usr
Resolved 26 packages in 1.00s
Prepared 15 packages in 49.87s
Uninstalled 2 packages in 21ms
Installed 15 packages in 270ms
 + nvidia-cublas-cu11==11.11.3.6
 + nvidia-cuda-cupti-cu11==11.8.87
 + nvidia-cuda-nvrtc-cu11==11.8.89
 + nvidia-cuda-runtime-cu11==11.8.89
 + nvidia-cudnn-cu11==9.1.0.70
 + nvidia-cufft-cu11==10.9.0.58
 + nvidia-curand-cu11==10.3.0.86
 + nvidia-cusolver-cu11==11.4.1.48
 + nvidia-cusparse-cu11==11.7.5.86
 + nvidia-nccl-cu11==2.21.5
 + nvidia-nvtx-cu11==11.8.86
 + torch==2.7.1+cu118
 - torchaudio==2.9.0+cpu (from https://download.pytorch.org/whl/cpu/torchaudio-2.9.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl)
 + torchaudio==2.7.1+cu118
 - torchvision==0.24.0+cpu (from https://download.pytorch.org/whl/cpu/torchvision-0.24.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl)
 + torchvision==0.22.1+cu118
 + triton==3

In [ ]:
!unzip "/content/sample dataset.zip"

Archive:  /content/sample dataset.zip
   creating: sample dataset/
   creating: sample dataset/Brian May/
  inflating: sample dataset/Brian May/brian may 10 young.jpg  
  inflating: sample dataset/Brian May/Brian may 2.jpg  
  inflating: sample dataset/Brian May/brian may 3.jpg  
  inflating: sample dataset/Brian May/brian may 4.jpg  
  inflating: sample dataset/Brian May/brian may 5 young.jpg  
  inflating: sample dataset/Brian May/Brian may 6.jpg  
  inflating: sample dataset/Brian May/brian may 7 young.jpg  
  inflating: sample dataset/Brian May/brian may 8 young.jpg  
  inflating: sample dataset/Brian May/brian may 9.jpg  
  inflating: sample dataset/Brian May/Brian may.jpg  
   creating: sample dataset/Carlos Santana/
  inflating: sample dataset/Carlos Santana/carlos santana 10.jpg  
  inflating: sample dataset/Carlos Santana/carlos santana 2.jpg  
  inflating: sample dataset/Carlos Santana/carlos santana 3 young.jpg  
  inflating: sample dataset/Carlos Santana/carlos santana 4.jp

In [ ]:
# enroll.py -> for labelled faces

import os
import cv2
import numpy as np
import pandas as pd
import pickle
from insightface.app import FaceAnalysis
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Initialize InsightFace
# -----------------------------
app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.5)
print("InsightFace model ready!")

# -----------------------------
# Dataset folder
# -----------------------------
DATASET_DIR = "/content/sample dataset"

# Storage
person_to_embeddings = {}
person_to_paths = {}

# -----------------------------
# Detect faces
# -----------------------------
def detect_faces(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image: {img_path}")
        return []

    if len(img.shape) == 2 or img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    faces = app.get(img)

    if len(faces) == 0:
        print(f"No face detected in {img_path}")

    return faces


# -----------------------------
# Walk dataset
# -----------------------------
for root, dirs, files in os.walk(DATASET_DIR):
    for file_name in files:

        img_path = os.path.join(root, file_name)
        if not os.path.isfile(img_path):
            continue

        person_name = os.path.basename(root)

        faces = detect_faces(img_path)
        if len(faces) == 0:
            continue

        # Take largest face
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))

        embedding = face.embedding
        embedding = embedding / np.linalg.norm(embedding)

        if person_name not in person_to_embeddings:
            person_to_embeddings[person_name] = []
            person_to_paths[person_name] = []

        person_to_embeddings[person_name].append(embedding)
        person_to_paths[person_name].append(img_path)


# -----------------------------
# OUTLIER REMOVAL
# -----------------------------
clean_person_embeddings = {}
clean_person_paths = {}

for person, emb_list in person_to_embeddings.items():

    if len(emb_list) < 3:
        clean_person_embeddings[person] = emb_list
        clean_person_paths[person] = person_to_paths[person]
        continue

    E = np.array(emb_list)
    sim_matrix = cosine_similarity(E, E)
    mean_sims = np.mean(sim_matrix, axis=1)

    threshold = np.mean(mean_sims) - 1.5 * np.std(mean_sims)

    filtered_embs = []
    filtered_paths = []

    for i in range(len(emb_list)):
        if mean_sims[i] >= threshold:
            filtered_embs.append(emb_list[i])
            filtered_paths.append(person_to_paths[person][i])

    print(f"{person}: kept {len(filtered_embs)}/{len(emb_list)} embeddings")

    clean_person_embeddings[person] = filtered_embs
    clean_person_paths[person] = filtered_paths


# -----------------------------
# BUILD FINAL DATABASE STRUCTURE
# -----------------------------
database = {}
person_stats = {}

for person, emb_list in clean_person_embeddings.items():

    if len(emb_list) == 0:
        continue

    E = np.array(emb_list)

    # ---- Centroid
    centroid = np.mean(E, axis=0)
    centroid = centroid / np.linalg.norm(centroid)

    # ---- Intra-class stats
    sims = []
    for i in range(len(E)):
        for j in range(i+1, len(E)):
            sims.append(np.dot(E[i], E[j]))

    if len(sims) > 0:
        intra_mean = float(np.mean(sims))
        intra_std  = float(np.std(sims))
    else:
        intra_mean = None
        intra_std  = None

    person_stats[person] = {
        "intra_mean": intra_mean,
        "intra_std": intra_std,
        "num_samples": len(E)
    }

    print(person, person_stats[person])

    database[person] = {
        "embeddings": emb_list,
        "paths": clean_person_paths[person],
        "centroid": centroid,
        "stats": person_stats[person]
    }


# -----------------------------
# Save database
# -----------------------------
output_file = "/content/image_face_embeddings_data.pkl"

with open(output_file, "wb") as f:
    pickle.dump(database, f)

print(f"\nSaved cleaned database to {output_file}")


# -----------------------------
# Optional inspection
# -----------------------------
rows = []

for person in database:
    for emb, path in zip(database[person]["embeddings"], database[person]["paths"]):
        rows.append({
            "label": person,
            "path": path
        })

faces_df = pd.DataFrame(rows)
print(faces_df.head())


download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 78300.98KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
InsightFace 

In [ ]:
with open("/content/image_face_embeddings_data.pkl", "rb") as f:
    database = pickle.load(f)

print("Identities:", list(database.keys()))

for person in database:
    print(f"\n{person}")
    print("Num embeddings:", len(database[person]["embeddings"]))
    print("Centroid shape:", database[person]["centroid"].shape)
    print("Stats:", database[person]["stats"])


Identities: ['Brian May', 'David Bowie', 'Frank Zappa', 'Carlos Santana', 'Kurt Cobain', 'Lady Gaga', 'George Michael', 'Ozzy Osbourne', 'Serj Tankian', 'Freddie Mercury']

Brian May
Num embeddings: 9
Centroid shape: (512,)
Stats: {'intra_mean': 0.5693973302841187, 'intra_std': 0.12432354688644409, 'num_samples': 9}

David Bowie
Num embeddings: 10
Centroid shape: (512,)
Stats: {'intra_mean': 0.598200798034668, 'intra_std': 0.08486068993806839, 'num_samples': 10}

Frank Zappa
Num embeddings: 7
Centroid shape: (512,)
Stats: {'intra_mean': 0.7211036682128906, 'intra_std': 0.04032508283853531, 'num_samples': 7}

Carlos Santana
Num embeddings: 9
Centroid shape: (512,)
Stats: {'intra_mean': 0.5159569978713989, 'intra_std': 0.10847688466310501, 'num_samples': 9}

Kurt Cobain
Num embeddings: 9
Centroid shape: (512,)
Stats: {'intra_mean': 0.5658707618713379, 'intra_std': 0.06029493734240532, 'num_samples': 9}

Lady Gaga
Num embeddings: 6
Centroid shape: (512,)
Stats: {'intra_mean': 0.4803092181

In [ ]:
labels = []
embeddings = []

for person in database:
    for emb in database[person]["embeddings"]:
        labels.append(person)
        embeddings.append(emb)

print("Total embeddings:", len(embeddings))
print("Embedding shape:", np.array(embeddings).shape)


Total embeddings: 84
Embedding shape: (84, 512)


Video embeddings extraction

In [ ]:
import cv2
import numpy as np
import pickle
import os
from insightface.app import FaceAnalysis
from tqdm.notebook import tqdm

# -------------------------------
# Initialize InsightFace
# -------------------------------
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(320, 320), det_thresh=0.4)
print("✅ InsightFace buffalo_l ready")

# -------------------------------
# Video path
# -------------------------------
VIDEO_PATH = "/content/David Bowie on Balancing Emotion and Intelligence When Creating.mp4"

# -------------------------------
# Generate UNIQUE pickle per video
# -------------------------------
video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_PICKLE = f"/content/{video_name}_enrollment.pkl"

print(f"📦 Enrollment output: {OUTPUT_PICKLE}")

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError("❌ Could not open video")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# -------------------------------
# Storage (fresh every run)
# -------------------------------
embeddings = []
frames_info = []

# -------------------------------
# Process video
# -------------------------------
pbar = tqdm(total=total_frames, desc="Enrolling video")

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    faces = app.get(frame_rgb)

    for face in faces:
        if face.normed_embedding is None:
            continue

        embeddings.append(face.normed_embedding)
        frames_info.append({
            "frame_idx": frame_idx,
            "bbox": face.bbox.astype(int)
        })

    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()

# -------------------------------
# Safety checks
# -------------------------------
if len(embeddings) == 0:
    raise ValueError("⚠ No faces detected in the video!")

embeddings = np.array(embeddings)

print(f"🧠 Total embeddings saved: {len(embeddings)}")
print(f"📏 Mean embedding norm: {np.mean(np.linalg.norm(embeddings, axis=1)):.3f}")

# -------------------------------
# Save pickle
# -------------------------------
with open(OUTPUT_PICKLE, "wb") as f:
    pickle.dump({
        "video_path": VIDEO_PATH,
        "total_frames": total_frames,
        "embeddings": embeddings,
        "frames_info": frames_info
    }, f)

print(f"✅ Enrollment complete. Saved to {OUTPUT_PICKLE}")


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (320, 320)
✅ InsightFac

ValueError: ❌ Could not open video